# Sổ tay 2 — Thăm dò mô hình ngôn ngữ (zero-shot probing)

Notebook này đo xem một mô hình ngôn ngữ (LM) còn làm đúng **tác vụ** trên câu
phương ngữ hay không — không chỉ đo độ "quen" (perplexity). Cách tiếp cận lặp
lại logic của `src/probe_models.py` trong codebase nghiên cứu: với mỗi câu, ta
chấm điểm log-probability của **mỗi nhãn ứng viên** rồi lấy softmax để đọc ra
nhãn dự đoán và độ tin cậy.

## Mục tiêu học tập

1. Phân biệt hai loại probing: perplexity (độ quen) vs. zero-shot task probing (đúng/sai + tin cậy).
2. Viết được chấm điểm log-probability của một completion và softmax theo nhãn.
3. Đo **accuracy** và **confidence** trên chuẩn vs. phương ngữ với ba LM.
4. Đo **confidence erosion** — độ tin cậy giảm bao nhiêu khi đổi chuẩn → dialect.
5. Giải thích vì sao accuracy tuyệt đối thấp không loại trừ việc đo được degradation.

## Tham chiếu

Code gốc trong codebase: `src/probe_models.py` (chạy local LM) và
`src/probe_openai.py` (chạy GPT-4o qua API). Cả hai dùng chung logic: build
prompt → chấm điểm mỗi nhãn ứng viên → softmax → nhãn dự đoán + confidence.
Notebook này rút logic cốt lõi ra để học viên chạy được trên 3 LM nhỏ.

## Công thức

Với prompt `x` và một nhãn ứng viên `y` (được bọc thành JSON như `{"label":"Anger"}`):

```text
seq_logprob(y | x) = Σ log p(y_t | x, y_<t)
avg_logprob(y | x) = seq_logprob / |y|
p(label | x) = softmax(avg_logprob) trên tập nhãn ứng viên
prediction = argmax_label p(label | x)
confidence  = p(prediction | x)
```

**Degradation (accuracy)** = acc(standard) − acc(dialect).
**Confidence erosion** = mean_confidence(standard) − mean_confidence(dialect).

Lợi thế so với perplexity: metric này gắn với **tác vụ** (đúng/sai nhãn), nên
dễ diễn giải với người làm ứng dụng. Hạn chế: chỉ áp dụng cho task phân loại
có nhãn cố định (MCQA, NLI, SENT). QA là sinh tự do, để phần mở rộng.

In [ ]:
from pathlib import Path
import sys

# Detect project root whether we run from notebooks/ or project root.
candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next((p for p in candidates if (p / "data" / "train_240.jsonl").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Run the notebook from the project root or notebooks/ directory.")
sys.path.insert(0, str(ROOT / "src"))
print(f"Project root: {ROOT}")

In [ ]:
import subprocess
import sys

INSTALL_DEPS = False
if INSTALL_DEPS:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers>=4.45,<5", "torch>=2.2,<3", "sentencepiece>=0.2,<1",
    ])

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from vialect_seas.data import DIALECTS, TASKS, load_jsonl
from vialect_seas.prompting import (
    LABEL_CANDIDATES, build_task_prompt, candidate_completion,
    is_classification, normalize_label, parse_prediction, gold_label,
)
from vialect_seas.probing import (
    DEFAULT_MODELS, load_text_generator, generate,
    score_completion, score_label_distribution,
    softmax_scores, probe_classification_rows,
)

sns.set_theme(style="whitegrid", context="notebook")
train = load_jsonl(ROOT / "data" / "train_240.jsonl")
test = load_jsonl(ROOT / "data" / "test_300.jsonl")
print(f"train={len(train)}  test={len(test)}")
print("Classification tasks (có nhãn cố định):",
      [t for t in TASKS if is_classification(t)])

### STUDENT TASK 0 — Kế hoạch thí nghiệm (preregistration)

Trước khi chạy mô hình, ghi dự đoán có hướng. Giữ dự đoán này khi xem kết quả.

In [ ]:
# HINT: RQ nên nêu rõ (1) input/population, (2) yếu tố so sánh, (3) metric.
"""Your code here"""
TEAM_NAME = "TODO"
PROBE_RQ = "TODO: ví dụ - ba LM có cùng suy giảm accuracy khi đổi chuẩn → dialect không?"
# Dự đoán thứ tự dialect từ ít → nhiều degradation.
EXPECTED_DIALECT_ORDER = ["TODO", "TODO", "TODO"]
# Dự đoán: confidence erosion có cùng hướng với accuracy degradation không?
PRED_CONFIDENCE_EROSSION = "TODO: cùng hướng / ngược hướng / không liên quan"

print({
    "team": TEAM_NAME,
    "rq": PROBE_RQ,
    "expected_order": EXPECTED_DIALECT_ORDER,
    "pred_confidence": PRED_CONFIDENCE_EROSSION,
})

## Ba mô hình ngôn ngữ

| Mô hình | Đặc điểm |
|---|---|
| `Qwen/Qwen2.5-0.5B` | multilingual base LM, scorer nhỏ |
| `bigscience/bloom-560m` | multilingual causal LM, tokenizer khác Qwen |
| `VietAI/gpt-neo-1.3B-vietnamese-news` | LM chuyên biệt tiếng Việt / tin tức |

Ba mô hình có tokenizer và pretraining corpus khác nhau. Accuracy tuyệt đối sẽ
thấp (zero-shot, mô hình nhỏ), nhưng ta quan tâm **paired degradation**
(chuẩn → dialect) bên trong từng mô hình, không phải điểm tuyệt đối.

## Chuẩn bị subset probing

Chỉ lấy task phân loại (MCQA, NLI, SENT) vì có nhãn ứng viên cố định. Lấy cân
bằng: `N_PER_CELL` câu/task/dialect từ train (giữ test nguyên để đánh giá sau).

In [ ]:
N_PER_CELL = 3  # 3 câu/task/dialect → 3 tasks × 3 dialect × 3 = 27 cặp × 2 variant = 54 probe
RUN_PROBING = False  # Đổi True khi có GPU/runtime phù hợp.

classification_train = train[train["task"].map(is_classification)].copy()
probe_frame = (
    classification_train
    .sort_values(["task", "target_dialect", "sample_id"])
    .groupby(["task", "target_dialect"], observed=True, group_keys=False)
    .head(N_PER_CELL)
    .reset_index(drop=True)
)
print("Probe rows:", len(probe_frame))
print(probe_frame.groupby(["task", "target_dialect"], observed=True).size().unstack(fill_value=0))

### STUDENT TASK 1 — Kiểm tra prompt và nhãn ứng viên

Trước khi chạy mô hình, in prompt cho một câu SENT và một câu NLI để xác nhận
prompt đúng định dạng. Kiểm tra nhãn gold đã được chuẩn hóa đúng.

In [ ]:
RUN_PROMPT_CHECK = False

if RUN_PROMPT_CHECK:
    # HINT 1: chọn 1 row SENT và 1 row NLI từ probe_frame.
    # HINT 2: in build_task_prompt(row, variant="dialect") và gold_label(row).
    # HINT 3: kiểm tra gold_label nằm trong LABEL_CANDIDATES[probe_task(row["task"])].
    """Your code here"""
    sent_row = None
    nli_row = None

    # SELF-CHECK
    assert sent_row is not None and nli_row is not None
    from vialect_seas.prompting import probe_task
    for r in [sent_row, nli_row]:
        g = gold_label(r)
        cands = LABEL_CANDIDATES[probe_task(r["task"])]
        assert g in cands, f"gold {g!r} không nằm trong {cands}"
    print("Prompt checks passed.")

## Chạy zero-shot probing

Code tải từng LM, chấm điểm mỗi nhãn ứng viên cho mỗi (row × variant), rồi giải
phóng VRAM trước khi tải LM tiếp theo. Bắt đầu với subset nhỏ; tăng `N_PER_CELL`
sau khi đã kiểm tra thời gian và output.

In [ ]:
from vialect_seas.prompting import probe_task
output_path = ROOT / "outputs" / "zero_shot_probe.csv"
output_path.parent.mkdir(exist_ok=True)

if RUN_PROBING:
    all_results = []
    for model_id in DEFAULT_MODELS:
        print(f"=== Probing {model_id} ===", flush=True)
        runner = load_text_generator(model_id)
        result = probe_classification_rows(probe_frame, runner, variants=("standard", "dialect"))
        all_results.append(result)
        # Giải phóng VRAM.
        import gc, torch
        del runner
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    probe_scores = pd.concat(all_results, ignore_index=True)
    probe_scores.to_csv(output_path, index=False)
    print("Saved", output_path)
elif output_path.exists():
    probe_scores = pd.read_csv(output_path)
    print("Loaded existing", output_path)
else:
    probe_scores = pd.DataFrame()
    print("Set RUN_PROBING=True (cần GPU) để chạy probing")

### Sanity checks

1. Mỗi mô hình có cùng số dòng (row × variant).
2. Không có confidence NaN hoặc ngoài [0, 1].
3. Số nhãn dự đoán.unique hợp lý (giới hạn trong candidates).

In [ ]:
if not probe_scores.empty:
    checks = probe_scores.groupby("model_id").agg(
        rows=("sample_id", "size"),
        mean_conf=("confidence", "mean"),
        min_conf=("confidence", "min"),
        max_conf=("confidence", "max"),
    )
    display(checks.round(4))
    assert checks["rows"].nunique() == 1, "Số dòng không khớp giữa các mô hình"
    assert checks["min_conf"].ge(0).all() and checks["max_conf"].le(1.001).all()

## Accuracy degradation theo dialect

`Drop = acc(standard) − acc(dialect)`; số dương nghĩa là mô hình **giảm đúng**
trên dialect. So sánh drop giữa ba mô hình và ba dialect.

In [ ]:
if not probe_scores.empty:
    acc = (
        probe_scores.groupby(["model_id", "target_dialect", "variant"], observed=True)
        .agg(accuracy=("correct", "mean"), n=("sample_id", "size"))
        .reset_index()
    )
    acc_pivot = acc.pivot_table(
        index=["model_id", "target_dialect"], columns="variant", values="accuracy"
    )
    acc_pivot["drop"] = acc_pivot["standard"] - acc_pivot["dialect"]

    fig, ax = plt.subplots(figsize=(10, 4.5))
    plot_acc = acc_pivot.reset_index().melt(
        id_vars=["model_id", "target_dialect"],
        value_vars=["standard", "dialect"], var_name="variant", value_name="accuracy"
    )
    sns.barplot(data=plot_acc, x="target_dialect", y="accuracy", hue="variant", ax=ax)
    ax.set(title="Accuracy: chuẩn vs. dialect (zero-shot)",
           xlabel="Dialect", ylabel="Accuracy")
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.show()
    display(acc_pivot.round(3))

### Insight (accuracy)

Viết 3–5 câu theo cấu trúc observation → evidence → caveat:

- Accuracy tuyệt đối có thể thấp (zero-shot, LM nhỏ) — điều đó không loại trừ
  việc đo được degradation có hướng.
- Dialect nào có drop lớn nhất? Có nhất quán giữa ba mô hình không?
- **Caveat:** subset nhỏ (N_PER_CELL=3) → khoảng tin cậy rộng; cần bootstrap
  theo source để báo cáo uncertainty.

## Confidence erosion

Ngoài accuracy, đo **độ tin cậy** (softmax của nhãn dự đoán). Confidence erosion
= mean_confidence(standard) − mean_confidence(dialect). Mô hình có thể giữ
accuracy nhưng giảm confidence — vẫn là dấu hiệu dialect "khó" với mô hình.

In [ ]:
if not probe_scores.empty:
    conf = (
        probe_scores.groupby(["model_id", "target_dialect", "variant"], observed=True)
        .agg(mean_confidence=("confidence", "mean"),
             mean_gold_prob=("gold_prob", "mean"))
        .reset_index()
    )
    conf_pivot = conf.pivot_table(
        index=["model_id", "target_dialect"], columns="variant", values="mean_confidence"
    )
    conf_pivot["erosion"] = conf_pivot["standard"] - conf_pivot["dialect"]

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    plot_conf = conf_pivot.reset_index().melt(
        id_vars=["model_id", "target_dialect"],
        value_vars=["standard", "dialect"], var_name="variant", value_name="mean_confidence"
    )
    sns.barplot(data=plot_conf, x="target_dialect", y="mean_confidence",
                hue="variant", ax=axes[0])
    axes[0].set(title="Confidence: chuẩn vs. dialect", xlabel="Dialect",
                ylabel="Mean confidence")
    axes[0].set_ylim(0, 1)

    sns.heatmap(conf_pivot["erosion"].unstack(), annot=True, fmt=".3f",
                cmap="RdBu_r", center=0, ax=axes[1])
    axes[1].set(title="Confidence erosion (chuẩn − dialect)",
                xlabel="Dialect", ylabel="Model")
    plt.tight_layout()
    plt.show()
    display(conf_pivot.round(3))

### Insight (confidence)

- Confidence erosion có cùng hướng với accuracy degradation không (theo dự đoán ở TASK 0)?
- Có trường hợp accuracy không đổi nhưng confidence giảm không? Ý nghĩa gì?
- **Caveat:** confidence từ softmax trên log-prob token có thể bị lệch bởi
  độ dài JSON của mỗi nhãn. Kiểm tra `num_tokens` nếu nghi ngờ.

In [ ]:
# So sánh theo task: degradation có khác giữa MCQA / NLI / SENT không?
if not probe_scores.empty:
    task_acc = (
        probe_scores.groupby(["model_id", "task", "variant"], observed=True)
        .agg(accuracy=("correct", "mean"))
        .reset_index()
    )
    task_acc_pivot = task_acc.pivot_table(
        index=["model_id", "task"], columns="variant", values="accuracy"
    )
    task_acc_pivot["drop"] = task_acc_pivot["standard"] - task_acc_pivot["dialect"]

    fig, ax = plt.subplots(figsize=(10, 4.5))
    sns.heatmap(task_acc_pivot["drop"].unstack(), annot=True, fmt=".3f",
                cmap="RdBu_r", center=0, ax=ax)
    ax.set(title="Accuracy drop theo task (chuẩn − dialect)",
           xlabel="Task", ylabel="Model")
    plt.tight_layout()
    plt.show()
    display(task_acc_pivot.round(3))

### Insight (theo task)

Xác định task nào tạo degradation lớn nhất. Với MCQA, kiểm tra liệu rewrite
có giữ đủ 4 lựa chọn và đáp án đúng không (xem Notebook 1). Với NLI, premise
không đổi — chỉ hypothesis bị dialect hóa — nên degradation cô lập được yếu tố
hypothesis.

## Bài tập mở

1. Tăng `N_PER_CELL` và tính bootstrap CI theo `sample_id` (resample theo source).
2. So sánh thứ tự khó ở đây với degradation trong Notebook 1 (9 mô hình lớn).
   **Correlation ≠ causation** — giải thích vì sao.
3. QA là task sinh tự do. Dùng `generate()` để sinh câu trả lời, rồi tính
   exact-match / contains-match với gold. So sánh chuẩn vs. dialect.
4. Thử `normalized_direct`: thay `dialect_text` bằng `normalized_text` từ
   Notebook 3, xem accuracy/confidence có phục hồi không.
5. Kiểm tra 10 câu có confidence erosion lớn nhất; phân loại nguyên nhân
   (từ vựng, ngữ pháp, độ dài).

In [ ]:
# STUDENT TASK 2 (EXTENSION) — Một thí nghiệm probing mở.
RUN_STUDENT_PROBE = False

def student_probe_analysis(scores):
    # HINT 1: chọn một biến chưa thử (task subset, normalized_direct, QA generative).
    # HINT 2: resample theo sample_id, giữ ba dialect của source trong cluster.
    # HINT 3: báo mean, 95% interval, n và aggregation unit.
    """Your code here"""
    return None

if RUN_STUDENT_PROBE:
    result = student_probe_analysis(probe_scores)
    if result is None:
        raise NotImplementedError("Hoàn thành student_probe_analysis trước khi bật cờ")
    # SELF-CHECK: bảng phải có model × dialect và interval hợp lệ.
    required = {"model_id", "target_dialect", "mean", "ci_low", "ci_high", "n_sources"}
    assert required.issubset(result.columns)
    assert (result["ci_low"] <= result["mean"]).all()
    assert (result["mean"] <= result["ci_high"]).all()
    display(result)